In [1]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


### Cell 2: download and look at the raw data

In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
ds = fetch_ucirepo(id=601)  # AI4I 2020 Predictive Maintenance
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### Cell 3: rename columns and keep only what we need

In [3]:
# Cell 3: Clean data and encode 'type' as numeric ordinal
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]
rename_map = {
    "Type": "type",
    "Air temperature": "air_temp_k",
    "Process temperature": "process_temp_k",
    "Rotational speed": "rpm",
    "Torque": "torque_nm",
    "Tool wear": "tool_wear_min",
    "Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]
# Map product type to numeric so AutoML does not create a sparse matrix
type_mapping = {"L": 0, "M": 1, "H": 2}
df["type"] = df["type"].map(type_mapping).astype(int)
# Ensure machine_failure is integer
df["machine_failure"] = df["machine_failure"].astype(int)
print(df.head())

   type  air_temp_k  process_temp_k   rpm  torque_nm  tool_wear_min  \
0     1       298.1           308.6  1551       42.8              0   
1     0       298.2           308.7  1408       46.3              3   
2     0       298.1           308.5  1498       49.4              5   
3     0       298.2           308.6  1433       39.5              7   
4     0       298.2           308.7  1408       40.0              9   

   machine_failure  
0                0  
1                0  
2                0  
3                0  
4                0  


### Cell 4: quality checks

In [4]:
print("Missing values per column:\n", df.isna().sum())
print("\nData types:\n", df.dtypes)
print("\nTarget balance:\n", df["machine_failure"].value_counts(normalize=True).round(4))

Missing values per column:
 type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
 type                 int64
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance:
 machine_failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64


### Cell 5: save the clean file

In [5]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")

Saved (10000, 7) to data/ai4i_clean.csv


### Cell 6: connect to your workspace

In [6]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
SUBSCRIPTION_ID = "e662579a-7cf4-4800-9ab8-fae6381625a9" # From Studio top-right profile
RESOURCE_GROUP = "rg-mlab1-seerat24"
WORKSPACE = "mlw-lab1-seerat24"
ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: mlw-lab1-seerat24


### Cell 7: register the CSV

In [7]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

csv_asset = Data(
    name="ai4i-maintenance-csv",
    version="2",
    type=AssetTypes.URI_FILE,
    path="../data/ai4i_clean.csv",
    description="AI4I 2020 predictive maintenance, cleaned & all-numeric",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

Uploading ai4i_clean.csv (< 1 MB): 100%|██████████| 295k/295k [00:00<00:00, 10.9MB/s]




Registered: ai4i-maintenance-csv version 2


In [8]:
import mltable

# Generate the MLTable configuration
tbl = mltable.from_delimited_files(paths=[{"file": "../data/ai4i_clean.csv"}])

# Save the folder containing the MLTable file
tbl.save("../data/ai4i-maintenance-table")
print("MLTable generated successfully!")

MLTable generated successfully!


In [11]:
import pandas as pd
import mltable
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# 1. Preview 7 columns and data types
df = pd.read_csv("../data/ai4i_clean.csv")
print("=== 7 COLUMNS PREVIEW ===")
display(df.head())
print("\n=== DATA TYPES ===")
print(df.dtypes[['type', 'machine_failure']])

# 2. Re-create the MLTable package pointing locally
mltable_dir = "../data/ai4i-maintenance-table"
tbl = mltable.from_delimited_files(paths=[{"file": "./ai4i_clean.csv"}])
tbl.save(mltable_dir)
df.to_csv(f"{mltable_dir}/ai4i_clean.csv", index=False)

# 3. Register as version 3
asset = Data(
    path=mltable_dir,
    type=AssetTypes.MLTABLE,
    description="predictive maintenance mltable for AutoML",
    name="ai4i-maintenance-table"
)

ml_client.data.create_or_update(asset)
print("\nSuccessfully registered ai4i-maintenance-table version 6!")

=== 7 COLUMNS PREVIEW ===


,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,1,298.1,308.6,1551,42.8,0,0
1,0,298.2,308.7,1408,46.3,3,0
2,0,298.1,308.5,1498,49.4,5,0
3,0,298.2,308.6,1433,39.5,7,0
4,0,298.2,308.7,1408,40.0,9,0



=== DATA TYPES ===
type               int64
machine_failure    int64
dtype: object

Successfully registered ai4i-maintenance-table version 6!


In [14]:
import os
import pandas as pd
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# 1. Prepare directory and save files
table_folder = "../data/ai4i-dense-clean"
os.makedirs(table_folder, exist_ok=True)

df = pd.read_csv("../data/ai4i_clean.csv")
df.to_csv(os.path.join(table_folder, "ai4i_clean.csv"), index=False)

# 2. MLTable definition
mltable_content = """paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      header: all_files_same_headers
      encoding: utf8
"""

with open(os.path.join(table_folder, "MLTable"), "w") as f:
    f.write(mltable_content)

# 3. Register under a fresh asset name
asset_dense = Data(
    path=table_folder,
    type=AssetTypes.MLTABLE,
    description="Dense predictive maintenance dataset for KNN AutoML",
    name="ai4i-maintenance-dense",
    version="1"
)

ml_client.data.create_or_update(asset_dense)
print("ai4i-maintenance-dense version 1 registered successfully!")

Uploading ai4i-dense-clean (0.3 MBs): 100%|██████████| 295094/295094 [00:00<00:00, 4627417.78it/s]




ai4i-maintenance-dense version 1 registered successfully!
